### Fiass Vector Store

Open Source Vector store developed by facebook

Faiss is a library for efficient similarity search and clustering of dense vectors

Faiss contains several methods for similarity search. It assumes that the instances are represented as vectors and are identified by an integer, and that the vectors can be compared with L2 (Euclidean) distances or dot products. Vectors that are similar to a query vector are those that have the lowest L2 distance or the highest dot product with the query vector. It also supports cosine similarity, since this is a dot product on normalized vectors.

In [1]:
from dotenv import load_dotenv
import numpy as np

In [31]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_classic.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

load_dotenv()


True

In [4]:
## Data Ingestion

sample_data = """
The debate over who reigns as football’s greatest player of all time (GOAT) spans decades, tactical revolutions, and iconic international tournaments. While football fans often favor candidates based on generation or playing style, a select group of legendary figures stands above the rest due to their extraordinary skill, unmatched longevity, and silver-tier achievements on the pitch.

Lionel Messi is widely regarded by many as the most complete attacking footballer in history. Emerging from Barcelona’s famed La Masia academy, the Argentine playmaker defined an era with his low center of gravity, close ball control, and astonishing goal creation. Messi holds a record eight Ballon d'Or awards and is the most decorated player in modern football history. His career reached its pinnacle in Qatar, where he captained Argentina to victory at the 2022 FIFA World Cup, solidifying his status alongside the sport's ultimate icons.

Cristiano Ronaldo represents the pinnacle of athletic perfection, goal-scoring efficiency, and mental fortitude. Rising to prominence with Manchester United before breaking scoring records at Real Madrid, Ronaldo evolved from a dynamic, skill-heavy winger into a lethal central forward. He is the all-time leading goalscorer in official men's football history with nearly 1,000 career goals, alongside five Ballon d'Or trophies and five UEFA Champions League titles. Furthermore, Ronaldo captained Portugal to their first major international triumph at UEFA Euro 2016.

Pelé remains the original global benchmark for footballing greatness. The Brazilian forward burst onto the international stage as a 17-year-old at the 1958 FIFA World Cup and went on to become the only player in history to win three World Cups (1958, 1962, and 1970). Playing the majority of his club career with Santos FC, Pelé netted over 1,000 career goals across official and exhibition matches. Beyond his statistics, he was a cultural pioneer who popularized the global phrase "The Beautiful Game" (O Jogo Bonito).

Diego Maradona possessed a unique blend of raw genius, emotional charisma, and incomparable dribbling ability. The Argentine maestro captured the football world’s imagination with his individual brilliance, most famously during the 1986 FIFA World Cup in Mexico. In Argentina’s quarter-final victory against England, Maradona scored two of history’s most famous goals: the controversial "Hand of God" and the "Goal of the Century," where he slalomed past six opponents from his own half. At the club level, he achieved immortal status by leading underdogs SSC Napoli to their first two Serie A titles in 1987 and 1990.

Johan Cruyff transformed football not only as an elite player but as the intellectual architect of modern tactics. As the engine of the Netherlands national team and Ajax during the 1970s, Cruyff pioneered "Total Football"—a fluid tactical system where outfield players dynamically interchanged positions. A three-time Ballon d'Or winner and three-time European Cup champion, Cruyff's positional mastery and vision laid the philosophical groundwork for modern European tactics at both Ajax and FC Barcelona.

Zinedine Zidane epitomized elegance, balance, and big-match composure in midfield. The French playmaker possessed immaculate control and turning ability, steering France to FIFA World Cup glory in 1998 with two header goals in the final and lifting UEFA Euro 2000. Known for producing iconic moments under pressure, Zidane scored one of the greatest goals in European history—a left-footed volley in the 2002 UEFA Champions League final for Real Madrid—firmly securing his place among the game's immortal playmakers.
"""

In [20]:
from langchain_community.document_loaders import DirectoryLoader

## Load all the files in the data folder with the help of DirectoryLoader module

dir_loader = DirectoryLoader(
    "../0-DataIngestionParsing/data/text_files",
    glob="**/Goats.txt", ## regular expression to match files
    loader_cls= TextLoader, ## loader class to use
    loader_kwargs= {'encoding': 'utf-8'},
    show_progress= True
)

documents = dir_loader.load()


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap = 60,
    length_function=len,
    separators=["\n\n","\n", " "]
)

chunks = text_splitter.split_documents(documents)

print(f"Created chunks - {len(chunks)}")
chunks

100%|██████████| 1/1 [00:00<00:00, 135.17it/s]

Created chunks - 8


[Document(metadata={'source': '..\\0-DataIngestionParsing\\data\\text_files\\Goats.txt'}, page_content='The debate over who reigns as football’s greatest player of all time (GOAT) spans decades, tactical revolutions, and iconic international tournaments. While football fans often favor candidates based on generation or playing style, a select group of legendary figures stands above the rest due to their extraordinary skill, unmatched longevity, and silver-tier achievements on the pitch.'),
 Document(metadata={'source': '..\\0-DataIngestionParsing\\data\\text_files\\Goats.txt'}, page_content="Lionel Messi is widely regarded by many as the most complete attacking footballer in history. Emerging from Barcelona’s famed La Masia academy, the Argentine playmaker defined an era with his low center of gravity, close ball control, and astonishing goal creation. Messi holds a record eight Ballon d'Or awards and is the most decorated player in modern football history. His career reached its pinna

In [21]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=model_name
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1782.50it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

###

### Create FIASS vector store

In [22]:
fiass_vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"Total vectors created with FAISS vector store is {fiass_vector_store.index.ntotal} vectors")

Total vectors created with FAISS vector store is 8 vectors


In [23]:
## Save vector store

fiass_vector_store.save_local("Faiss_vectore_store")

In [ ]:
## Loading the vector store

# Structure of vector store
# ├── index.faiss   <-- C++ Coded: Pure vector index (floating-point numbers)
# └── index.pkl     <-- Python Coded: Map linking Vector IDs -> Text Chunks & Metadat

loaded_vector_store = FAISS.load_local(
    "Faiss_vectore_store",
    embeddings=embeddings, ## FAISS needs an embedding model to convert your incoming text query into a vector before comparing it against the index.
    allow_dangerous_deserialization=True ## we are saying to trust the pickle files
)

print(f"loaded vectors - {loaded_vector_store.index.ntotal}")

loaded vectors - 8


In [ ]:
## Similarity search

query = "Tell me more about messi?"

results = loaded_vector_store.similarity_search(query,k=3)

# results = loaded_vector_store.similarity_search(query,k=3,
#                                                 filter=<apply any filters>)

for i, doc in enumerate(results):
    print(f"\n {i+1} Source: {doc.metadata}")
    print(f"\n Content: {doc.page_content[:300]}...")


 1 Source: {'source': '..\\0-DataIngestionParsing\\data\\text_files\\Goats.txt'}

 Content: Lionel Messi is widely regarded by many as the most complete attacking footballer in history. Emerging from Barcelona’s famed La Masia academy, the Argentine playmaker defined an era with his low center of gravity, close ball control, and astonishing goal creation. Messi holds a record eight Ballon ...

 2 Source: {'source': '..\\0-DataIngestionParsing\\data\\text_files\\Goats.txt'}

 Content: Cristiano Ronaldo represents the pinnacle of athletic perfection, goal-scoring efficiency, and mental fortitude. Rising to prominence with Manchester United before breaking scoring records at Real Madrid, Ronaldo evolved from a dynamic, skill-heavy winger into a lethal central forward. He is the all...

 3 Source: {'source': '..\\0-DataIngestionParsing\\data\\text_files\\Goats.txt'}

 Content: Diego Maradona possessed a unique blend of raw genius, emotional charisma, and incomparable dribbling ability. Th

### Build RAG chain with FIASS

In [ ]:
## LLM GROQ 

from langchain.chat_models import init_chat_model

# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

groq_llm = init_chat_model(model="groq:llama-3.3-70b-versatile")

groq_llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000215D7CEA900>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000215D7CEEDB0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [32]:
simple_prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

## 📌 Summary Comparison: Raw FAISS Vector Store vs. VectorStoreRetriever

| Feature | Raw `FAISS` Vector Store | `VectorStoreRetriever` |
| :--- | :--- | :--- |
| **Primary Job** | Stores vectors, indexes data, and performs vector distance math | Acts as a standardized data fetching interface for chains |
| **Input Method** | Requires explicit search method calls (`similarity_search()`) | Accepts plain text queries directly via `.invoke("query")` |
| **Pipeline Integration** | Requires manual Python data-formatting steps | Direct LCEL pipe support (`retriever \| prompt \| llm`) |
| **Advanced Search** | Basic k-NN similarity search | Supports MMR, thresholding, compression, & hybrid search |

---

### 💡 Key Takeaway for RAG Pipelines
* Use **`FAISS`** directly when building, saving, loading, or inspecting the vector index.
* Convert to **`VectorStoreRetriever`** (`vectorstore.as_retriever()`) when wiring your store into an automated LangChain / LLM generation pipeline.

In [34]:
fiass_retriever = fiass_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [35]:
fiass_retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000215D7CD1850>, search_kwargs={'k': 3})

In [36]:
from typing import List
def format_docs(docs:List[Document]) -> str:
    # Format documents for insertion into prompt
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [37]:
simple_rag_chain = (
    {
        "context": fiass_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | groq_llm
    | StrOutputParser()
)

In [38]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000215D7CD1850>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_in

In [39]:
## Conversational RAG using FAISS 

## create a prompt that includes the chat history

contextualize_system_prompt = """
You are a query reformulator. Your sole job is to rewrite the user's latest question into a standalone, clear search query using the provided chat history for context.

STRICT INSTRUCTIONS:
1. Do NOT answer the question under any circumstances.
2. Resolve all ambiguous pronouns (e.g., "it", "they", "this", "that") using the chat history.
3. If the question is already standalone and clear, return it word-for-word without any changes.
4. Do NOT add conversational filler like "Here is the rephrased question:" or introductory
"""

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt), ## place holder for preserving chat history
    ("placeholder", "{chat_history}"),
    ("human","{input}")
])

#### More about the below lambda statement

📌 Deep Dive: The Role of lambda in LCEL Chains
In LangChain Expression Language (LCEL), placing an inline lambda function inside a dictionary assignment step serves as an anonymous data-processing pipeline. It dynamically extracts the user query from the incoming dictionary, queries the vector database, formats the retrieved documents into a string, and assigns the result to a new dictionary key.

🔍 Execution Breakdown (Step-by-Step)

* #### State Dictionary Injection (x)

The parameter x represents the entire state dictionary entering that point in the LCEL pipe.

It contains all key-value pairs passed from the caller (such as the user's input question string).

* #### Key Extraction (x["input"])

The lambda accesses the specific string stored under the input key.

This isolates the user's raw question text so it can be processed independently.

* #### Vector Database Retrieval (fiass_retriever.invoke(...))

The extracted query string is passed directly into the FAISS retriever.

The retriever embeds the string into a vector, calculates similarity distances, and fetches the top matching document objects stored in the vector database.

* #### Context Formatting (format_docs(...))

The list of retrieved document objects is passed into a helper function.

This function extracts the raw text content from each document and joins them into a single continuous text string (often separated by line breaks).

* #### Dictionary Enrichment (context = ...)

The final formatted text string is returned by the lambda function.


When you use RunnablePassthrough.assign(...), you are asking LangChain to take the existing dictionary input, keep all original keys, and dynamically add a new key (in this case, "context") to that dictionary before passing it to the prompt.

The .assign() method appends this string under the new key "context" alongside the original "input" key.

In [ ]:
def create_conversational_RAG():
    return(
        RunnablePassthrough.assign(
            context = lambda x: format_docs(fiass_retriever.invoke(x["input"])) ## a lambda function is a small, anonymous function that is defined without a name and written in a single line of code. Unlike standard functions created using the def keyword
        )
        | contextualize_prompt
        | groq_llm
        | StrOutputParser()
    )

conversation_RAG = create_conversational_RAG()

In [41]:
conversation_RAG

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(fiass_retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='Chat

A Streaming RAG Chain is a Retrieval-Augmented Generation pipeline designed to output the generated response token-by-token in real time as the Large Language Model (LLM) creates it, rather than waiting for the entire generation process to finish before returning the answer.

In [42]:
## Streaming RAG Chain

streaming_RAG_chain = (
    {"context": fiass_retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | groq_llm
)

In [ ]:
# Test for different RAG Chains

def test_RAG_chains(question: str):
    print(f"The Question: {question}")
    print("+" * 100)

    # print("\n 1 - Simple RAG chain\n")
    # answer = simple_rag_chain.invoke(question)
    # print(f"The Answer from the LLM - {answer}")

    print("\n 3 - Streaming RAG chain\n")
    print("Answer: ", end="", flush=True) # "Answer: ": Prints a label to indicate that the model's response is starting, end="": Overrides Python’s default print() behavior (which adds a newline \n at the end). This keeps the cursor on the same line so incoming tokens print directly after "Answer:, flush=True: Forces Python to clear its output buffer immediately and render "Answer: " on the screen without waiting for more text to arrive.
    for chunk in streaming_RAG_chain.stream(question):
        print(chunk.content, end="", flush=True) # chunk.content: Extracts the actual text string from the incoming AIMessageChunk object returned by the model.
    print()



In [49]:
test_RAG_chains("can you tell me the difference between messi and ronaldo?")

The Question: can you tell me the difference between messi and ronaldo?
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

 1 - Simple RAG chain

The Answer from the LLM - Based on the provided context, the main differences between Messi and Ronaldo are:

1. **Playing style**: Messi is described as having a "low center of gravity, close ball control, and astonishing goal creation", while Ronaldo is known for his "athletic perfection, goal-scoring efficiency, and mental fortitude" and evolved from a "dynamic, skill-heavy winger" to a "lethal central forward".

2. **Awards and records**: Messi holds a record eight Ballon d'Or awards, while Ronaldo has five. Ronaldo is the all-time leading goalscorer in official men's football history with nearly 1,000 career goals.

3. **Career highlights**: Messi captained Argentina to victory at the 2022 FIFA World Cup, while Ronaldo captained Portugal to their first major international triumph at UEFA

In [53]:
## Conversational RAG

chat_history = []

prompt = "if you compare them, who do you think is the Goal Scorer of all time?"

chat_history.extend([
    HumanMessage(content=prompt),
    AIMessage(content="""
1. **Playing style**: Messi is described as having a "low center of gravity, close ball control, and astonishing goal creation", while Ronaldo is known for his "athletic perfection, goal-scoring efficiency, and mental fortitude" and evolved from a "dynamic, skill-heavy winger" to a "lethal central forward".

2. **Awards and records**: Messi holds a record eight Ballon d'Or awards, while Ronaldo has five. Ronaldo is the all-time leading goalscorer in official men's football history with nearly 1,000 career goals.

3. **Career highlights**: Messi captained Argentina to victory at the 2022 FIFA World Cup, while Ronaldo captained Portugal to their first major international triumph at UEFA Euro 2016.

4. **Team affiliations**: Messi emerged from Barcelona's La Masia academy, while Ronaldo rose to prominence with Manchester United and broke scoring records at Real Madrid.

 3 - Streaming RAG chain

Answer: Based on the provided context, the differences between Messi and Ronaldo can be summarized as follows:

1. **Playing Style**: Messi is described as having a "low center of gravity, close ball control, and astonishing goal creation", implying a more agile and creative playing style. In contrast, Ronaldo is portrayed as having evolved from a "dynamic, skill-heavy winger" into a "lethal central forward", suggesting a more athletic and goal-scoring focused approach.

2. **Achievements**: Messi has won a record eight Ballon d'Or awards and led Argentina to victory at the 2022 FIFA World Cup. Ronaldo, on the other hand, is the all-time leading goalscorer in official men's football history with nearly 1,000 career goals, has won five Ballon d'Or trophies, and led Portugal to their first major international triumph at UEFA Euro 2016.

3. **Career Path**: Messi emerged from Barcelona's La Masia academy, while Ronaldo rose to prominence with Manchester United before breaking scoring records at Real Madrid.

These differences highlight distinct aspects of their careers and playing styles, showcasing their unique contributions to the sport.
""")
])

answer = conversation_RAG.invoke({
    "input": prompt,
    "chat_history": chat_history
})

print(f"Q1: {prompt}")

answer1 = test_RAG_chains(answer)

print(f"A1 : {answer1}")


Q1: if you compare them, who do you think is the Goal Scorer of all time?
The Question: who do you think is the Goal Scorer of all time between Lionel Messi and Cristiano Ronaldo
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

 1 - Simple RAG chain

The Answer from the LLM - Based on the provided context, Cristiano Ronaldo is mentioned as the "all-time leading goalscorer in official men's football history with nearly 1,000 career goals", while Lionel Messi's goal-scoring record is not explicitly stated in terms of being the highest. Therefore, based on the information given, Cristiano Ronaldo would be considered the goal scorer of all time between the two.

 3 - Streaming RAG chain

Answer: Based on the provided context, Cristiano Ronaldo is considered the all-time leading goalscorer in official men's football history with nearly 1,000 career goals, whereas the context does not provide the exact number of career goals for Lionel Mes